# CPV VisDrone Training

**Phase 5 sanity:** `EPOCHS = 5`, `CONFIG = configs/yolov8n.yaml`  
**Phase 6 full:** `EPOCHS = 50`, change `CONFIG` per model run

Requires: GPU T4 x1, Internet ON, dataset `itsdreowo/cpv-visdrone5` attached.

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
REPO      = "https://github.com/its-dreOwO/CPV.git"
CONFIG    = "configs/yolov8n.yaml"   # yolov8m.yaml | rtdetr.yaml for Phase 6
EPOCHS    = 5                        # 5 = sanity, 50 = full
DATA_ROOT = "/kaggle/input/cpv-visdrone5"
DEVICE    = "cuda"

In [ ]:
import subprocess, os, sys

# Clone repo
if not os.path.exists("/kaggle/working/CPV"):
    subprocess.run(["git", "clone", REPO, "/kaggle/working/CPV"], check=True)
os.chdir("/kaggle/working/CPV")
sys.path.insert(0, "/kaggle/working/CPV")

# filterpy is not pre-installed on Kaggle (needed by KalmanTracker, not training)
# scikit-learn is pre-installed; install filterpy just in case
subprocess.run([sys.executable, "-m", "pip", "install", "filterpy>=1.4.5", "-q"], check=True)

print("Setup complete. CWD:", os.getcwd())

In [ ]:
# Verify GPU
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

In [ ]:
# Train
subprocess.run(
    [
        sys.executable, "scripts/train.py",
        "--config",    CONFIG,
        "--epochs",    str(EPOCHS),
        "--device",    DEVICE,
        "--data-root", DATA_ROOT,
    ],
    check=True,
)

In [ ]:
# Show training curves
import glob
from IPython.display import Image, display

for img in sorted(glob.glob("runs/*/results.png")):
    print(img)
    display(Image(img))

In [ ]:
# Copy best weights to /kaggle/working/ so they appear in notebook output
import shutil, pathlib

for pt in sorted(pathlib.Path("runs").glob("*/weights/best.pt")):
    dest = pathlib.Path("/kaggle/working") / pt.parent.parent.name / "best.pt"
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(pt, dest)
    print("Saved:", dest)